# Μάθημα 17 - Δημιουργία Τοπικών Πρακτόρων AI με Foundry Local και Qwen

Σε αυτό το σημειωματάριο δημιουργείτε έναν **τοπικό βοηθό μηχανικής** που εκτελείται εξ ολοκλήρου στον υπολογιστή σας. Δεν χρησιμοποιείται καμία εκτέλεση στο cloud σε κανένα σημείο. Ο βοηθός μπορεί:

1. **Να καλεί εργαλεία** μέσω κλήσης λειτουργιών Qwen μέσω Foundry Local.
2. **Να απαριθμεί και να διαβάζει αρχεία** μέσα σε έναν προστατευμένο κατάλογο έργου.
3. **Να αναλύει κώδικα** με απλά μετρικά.
4. **Να αναζητά τεκμηρίωση** με τοπικό RAG (Chroma).
5. **Να χρησιμοποιεί έναν τοπικό διακομιστή MCP** (παραλείπεται ομαλά αν δεν έχει ρυθμιστεί).

Ο κώδικας του πράκτορα είναι σχεδόν πανομοιότυπος με αυτόν των μαθημάτων στο cloud — μόνο το σημείο τελικού πελάτη μετακινείται από το cloud στο `localhost`.


## Ρύθμιση

Πριν εκτελέσετε αυτό το σημειωματάριο:

1. **Εγκαταστήστε το Microsoft Foundry Local** (δείτε την [τεκμηρίωση](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) για το λειτουργικό σας σύστημα).
2. **Κατεβάστε και ξεκινήστε ένα μοντέλο Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Εγκαταστήστε τα παρακάτω πακέτα Python.

Όλα εκτελούνται τοπικά. Μια μηχανή με ~8 GB RAM είναι ένα ρεαλιστικό ελάχιστο.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Σύνδεση στο Foundry Local

Το `FoundryLocalManager` κατεβάζει το μοντέλο αν χρειάζεται, ξεκινά την τοπική υπηρεσία και μας παρέχει ένα **endpoint συμβατό με το OpenAI**. Στη συνέχεια, δείχνουμε το τυπικό SDK του OpenAI σε αυτό. Το API key είναι ένας τοπικός δείκτης — δεν εμπλέκεται διαπίστευση cloud.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Τοπικά Εργαλεία (Απομονωμένες Λειτουργίες Αρχείων)

Δημιουργούμε ένα μικρό δείγμα έργου στον δίσκο και στη συνέχεια ορίζουμε εργαλεία που περιορίζονται στη ρίζα αυτού του έργου. Ο έλεγχος απομόνωσης έχει σημασία ακόμα και τοπικά: ένα εργαλείο που διαβάζει αυθαίρετες διαδρομές εκτελείται με τα δικαιώματα του χρήστη σας και μπορεί να αγγίξει οτιδήποτε μπορείτε κι εσείς.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Τοπικό RAG με Chroma

Ενσωματώνουμε ένα μικρό σύνολο αποσπασμάτων τεκμηρίωσης σε μια τοπική συλλογή Chroma. Το Chroma εκτελείται εντός της διαδικασίας και αποθηκεύει διανύσματα στον δίσκο — χωρίς διακομιστή, χωρίς σύννεφο. Το εργαλείο `search_docs` ανακτά τα πιο σχετικά αποσπάσματα για ένα ερώτημα.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Ο βρόχος κλήσης εργαλείων

Τώρα καταχωρούμε τα εργαλεία με το μοντέλο χρησιμοποιώντας το σχήμα εργαλείων OpenAI και εκτελούμε τον τυπικό βρόχο κλήσης εργαλείων — το μοντέλο ζητά ένα εργαλείο, το εκτελούμε τοπικά, τροφοδοτούμε το αποτέλεσμα πίσω και επαναλαμβάνουμε μέχρι το μοντέλο να παράγει μια τελική απάντηση. Η αξιόπιστη κλήση συναρτήσεων του Qwen είναι αυτό που κάνει αυτή τη διαδικασία να λειτουργεί στη συσκευή.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Τοπικό MCP (Προαιρετικό)

Το MCP είναι ένα μέσο μεταφοράς, όχι μια υπηρεσία cloud — ένας διακομιστής MCP μπορεί να τρέξει ως τοπική διεργασία μέσω `stdio`. Το κελί παρακάτω δείχνει πώς θα συνδεόσασταν σε έναν τοπικό διακομιστή MCP αν έχετε έναν ρυθμισμένο (για παράδειγμα έναν διακομιστή συστήματος αρχείων). Παρακάμπτει ομαλά όταν το `LOCAL_MCP_COMMAND` δεν είναι ορισμένο, ώστε το σημειωματάριο να τρέχει ολόκληρο χωρίς αυτό.

Σημείωση ασφαλείας: ένας τοπικός διακομιστής MCP τρέχει με τα δικαιώματα του χρήστη σας. Περιορίστε τον σε έναν κατάλογο έργου και επικυρώστε τα αποτελέσματά του πριν ενεργήσετε βάσει αυτών.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Περίληψη

Δημιουργήσατε έναν βοηθό μηχανικού που λειτουργεί εξ ολοκλήρου στη μηχανή σας:

- **Foundry Local** εξυπηρετούσε ένα μοντέλο **Qwen** πίσω από ένα endpoint συμβατό με OpenAI — ώστε ο κώδικας του πράκτορα να ταιριάζει με τα μαθήματα στο cloud.
- **Sandboxed tools** παρείχαν στον πράκτορα πρόσβαση σε αρχεία και ανάλυση κώδικα χωρίς να βγει από τον κατάλογο έργου.
- **Chroma** παρείχε **τοπικό RAG** πάνω στην τεκμηρίωση.
- **Local MCP** έδειχνε πώς να επαναχρησιμοποιείται το οικοσύστημα MCP εκτός σύνδεσης.

Δεν χρησιμοποιήθηκε ποτέ cloud inference.

### Πρόκληση

Επεκτείνετε τον τοπικό πράκτορα ώστε να:

1. **Εργάζεται με πολλαπλούς MCP servers** — συνδέστε έναν διακομιστή συστήματος αρχείων και έναν διακομιστή git και επιτρέψτε στον πράκτορα να επιλέξει μεταξύ αυτών.
2. **Χρησιμοποιεί τοπική μνήμη** — αποθηκεύστε ένα σύντομο ιστορικό συνομιλίας στον δίσκο ώστε ο βοηθός να θυμάται προηγούμενες εναλλαγές ανάμεσα σε επανεκκινήσεις του notebook.
3. **Υποστηρίζει τοπικό συντονισμό πολλαπλών πρακτόρων** — προσθέστε έναν δεύτερο τοπικό πράκτορα (π.χ. έναν κριτή) και κάντε τους δύο να συνεργαστούν σε μια εργασία.

Στο επόμενο μάθημα θα μάθετε πώς να ασφαλίζετε τους αναπτυγμένους πράκτορες AI.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
